In [ ]:
!pip install control

import control as sc

## Referencia rapida de comandos (Python / control)

> Nota: para este notebook se usara `python-control` con `import control as sc`.
> Los comandos de esta guia se invocan tipicamente como `sc.tf`, `sc.ss`, `sc.pole`, etc.

- **tic / toc**: par de funciones para medir tiempo transcurrido (estilo MATLAB). En Python suele hacerse con `time.time()` o `time.perf_counter()`.
- **time**: modulo para manejar tiempo; util para medir duracion de ejecucion y pausas.
- **tf**: crea un modelo en **funcion de transferencia**.
- **ss**: crea un modelo en **espacio de estados**.
- **zpk**: crea un modelo en forma **ceros-polos-ganancia**.
- **tf2ss**: convierte de funcion de transferencia a espacio de estados.
- **tf2zp**: convierte de funcion de transferencia a ceros, polos y ganancia.
- **ss2zp**: convierte de espacio de estados a ceros, polos y ganancia (equivalente comun: `ss2zpk`).
- **ss2tf**: convierte de espacio de estados a funcion de transferencia.
- **zp2ss**: convierte de ceros, polos y ganancia a espacio de estados.
- **zp2tf**: convierte de ceros, polos y ganancia a funcion de transferencia.
- **roots**: calcula las raices de un polinomio.
- **pole**: obtiene los polos del sistema.
- **zero**: obtiene los ceros del sistema.
- **poly**: construye un polinomio a partir de sus raices.
- **eig**: calcula valores propios (autovalores) de una matriz.
- **pzmap**: grafica polos y ceros en el plano complejo.
- **dcgain**: calcula la ganancia estatica (ganancia en $s=0$).
- **minreal**: elimina cancelaciones polo-cero y obtiene una realizacion minima.
- **subplot**: crea varias subgraficas en una misma figura.
- **feedback**: forma un lazo de realimentacion (abierto/cerrado).
- **tfdata**: extrae numerador y denominador de un modelo `tf`.

Mas informacion: https://python-control.readthedocs.io/en/0.10.1/index.html

## 3.3 Respuestas conceptuales

- **¿Qué es el polinomio característico de un sistema?**  
  Es el polinomio que se obtiene de $\det(sI-A)=0$ en espacio de estados (o del denominador de la función de transferencia en sistemas LTI). Sus raíces determinan la dinámica natural del sistema.

- **¿Qué son los valores propios de un sistema?**  
  Son los escalares $\lambda$ que satisfacen $Av=\lambda v$ para algún vector no nulo $v$. En control, corresponden a los polos del sistema (en realización mínima) y definen estabilidad, rapidez y tipo de respuesta.

- **¿Qué son el tiempo de estabilización y el sobre nivel porcentual en la respuesta al escalón?**  
  - **Tiempo de estabilización ($T_s$)**: tiempo que tarda la salida en entrar y permanecer dentro de una banda de tolerancia alrededor del valor final (típicamente $\pm2\%$ o $\pm5\%$).  
  - **Sobre nivel porcentual ($\%OS$)**: cuánto supera el pico máximo al valor final, en porcentaje:
  $$
  \%OS = \frac{y_{\max}-y_{\infty}}{y_{\infty}}\times 100
  $$
  Mide el grado de sobreimpulso de la respuesta transitoria.

## Modelo experimental y desarrollo 5.1

Supongamos que experimentamos con el sistema, el cual se encuentra en un punto de operacion estable, y que lo sometemos a un cambio en la senal de entrada $u(t)$. Esto produce un cambio en el punto de operacion del sistema y, como consecuencia, un cambio en la senal de respuesta $y(t)$.

Despues de un procedimiento matematico, con base en las senales experimentales, se obtiene para el sistema un modelo en forma de Funcion de Transferencia (FT):

$$
G(s)=\frac{k e^{-sT_m}}{(\tau_1 s+1)(\tau_2 s+1)}
=\frac{1.2\,e^{-39.2s}}{(59.7s+1)(22.3s+1)}
$$

Donde:
- $T_m$: tiempo muerto
- $k$: ganancia del sistema
- $\tau_1,\tau_2$: constantes de tiempo

Este modelo de segundo orden con tiempo muerto (MSOTM) tiene una respuesta al escalon que se superpone a la del sistema original. El tiempo muerto $T_m=39.2\,s$ desplaza en el tiempo la respuesta del modelo en esa cantidad.

### 5.1 Conversion FT -> EE (sin tiempo muerto)

Partimos del modelo sin tiempo muerto:

$$
G_0(s)=\frac{k}{(\tau_1 s+1)(\tau_2 s+1)}
=\frac{k}{\tau_1\tau_2 s^2+(\tau_1+\tau_2)s+1}
$$

Relacion entrada-salida:

$$
\left(\tau_1\tau_2 s^2+(\tau_1+\tau_2)s+1\right)Y(s)=kU(s)
$$

En tiempo continuo:

$$
\tau_1\tau_2\,\ddot y(t)+(\tau_1+\tau_2)\,\dot y(t)+y(t)=k\,u(t)
$$

Con coeficiente de la mayor derivada igual a 1:

$$
\ddot y(t)+\frac{\tau_1+\tau_2}{\tau_1\tau_2}\dot y(t)+\frac{1}{\tau_1\tau_2}y(t)=\frac{k}{\tau_1\tau_2}u(t)
$$

Usando Variables de Estado de Fase (VEF):

$$
x_1(t)=y(t),\quad x_2(t)=\dot y(t)
$$

Se obtiene:

$$
\dot x_1=x_2
$$

$$
\dot x_2=-\frac{1}{\tau_1\tau_2}x_1-\frac{\tau_1+\tau_2}{\tau_1\tau_2}x_2+\frac{k}{\tau_1\tau_2}u
$$

$$
y=x_1
$$

Forma matricial:

$$
\dot x=Ax+Bu,\quad y=Cx+Du
$$

$$
A=\begin{bmatrix}
0 & 1\\
-\frac{1}{\tau_1\tau_2} & -\frac{\tau_1+\tau_2}{\tau_1\tau_2}
\end{bmatrix},\quad
B=\begin{bmatrix}
0\\
\frac{k}{\tau_1\tau_2}
\end{bmatrix},\quad
C=\begin{bmatrix}1 & 0\end{bmatrix},\quad
D=\begin{bmatrix}0\end{bmatrix}
$$

Modelo de FT sin tiempo muerto:

$$
G_0(s)=\frac{k}{(\tau_1 s+1)(\tau_2 s+1)}
$$

Modelo de EE sin tiempo muerto:

$$
\dot x_1=x_2,\qquad
\dot x_2=-\frac{1}{\tau_1\tau_2}x_1-\frac{\tau_1+\tau_2}{\tau_1\tau_2}x_2+\frac{k}{\tau_1\tau_2}u,\qquad
y=x_1
$$

## Ejemplos minimos de uso con `control as sc`

En este notebook usaremos la libreria `control` con el alias `sc`.
Los siguientes ejemplos muestran la sintaxis base de los comandos mas usados.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import tf2zpk

# Modelo de ejemplo
num_ej = [1.2]
den_ej = np.polymul([59.7, 1], [22.3, 1]).tolist()
G_ej = sc.tf(num_ej, den_ej)

# Conversion FT -> EE: tf2ss devuelve un objeto StateSpace, se accede por atributos
G_ej_ss = sc.tf2ss(G_ej)
A_ej = G_ej_ss.A
B_ej = G_ej_ss.B
C_ej = G_ej_ss.C
D_ej = G_ej_ss.D

# Ceros, polos y ganancia usando scipy (tf2zpk) o control
Z_ej, P_ej, K_ej = tf2zpk(num_ej, den_ej)

print('Polos         :', sc.poles(G_ej))
print('Ceros         :', sc.zeros(G_ej))
print('Ganancia DC   :', sc.dcgain(G_ej))
print('Raices del den:', np.roots(den_ej))
print('Poly(polos)   :', np.poly(sc.poles(G_ej)))
print('Autovalores A :', np.linalg.eigvals(A_ej))

# Mapa de polos y ceros
sc.pzmap(G_ej, plot=True)
plt.title('Mapa de polos y ceros (ejemplo)')
plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.grid(True)
plt.show()

## 5.2 Implementacion en Python (sin tiempo muerto)

Ahora se define en Python el modelo sin tiempo muerto en sus dos formas:

- Funcion de transferencia: `Gptf = sc.tf(num, den)`
- Espacio de estado: `Mpss = sc.ss(A, B, C, D)`

Luego se comparan las respuestas al escalon de `0.23*Gptf` y `0.23*Mpss`, y se obtienen:

- Valor final de la respuesta (5.2.1)
- Tiempo de estabilizacion (5.2.2)

Como apoyo se usa `sc.step_info(...)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parametros del modelo sin tiempo muerto
k = 1.2
tau1 = 59.7
tau2 = 22.3

# FT: G0(s) = k / ((tau1*s+1)(tau2*s+1))
num = [k]
den = np.polymul([tau1, 1], [tau2, 1])

# EE en variables de estado de fase: x1=y, x2=dy/dt
A = np.array([
    [0, 1],
    [-1/(tau1*tau2), -(tau1+tau2)/(tau1*tau2)]
], dtype=float)
B = np.array([[0], [k/(tau1*tau2)]], dtype=float)
C = np.array([[1, 0]], dtype=float)
D = np.array([[0]], dtype=float)

# Modelos
Gptf = sc.tf(num, den)
Mpss = sc.ss(A, B, C, D)

print('Gptf =')
print(Gptf)
print('\nMpss =')
print(Mpss)

# Entrada escalon de amplitud 0.23
amp = 0.23
Gptf_amp = amp * Gptf
Mpss_amp = amp * Mpss

# Tiempo de simulacion
T = np.linspace(0, 500, 2000)

# Respuesta al escalon
T_tf, y_tf = sc.step_response(Gptf_amp, T)
T_ss, y_ss = sc.step_response(Mpss_amp, T)

# Graficas solicitadas (equivalente a step + figure + step)
plt.figure(figsize=(8, 4))
plt.plot(T_tf, y_tf)
plt.title('Respuesta al escalon: 0.23*Gptf')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.grid(True)

plt.figure(figsize=(8, 4))
plt.plot(T_ss, y_ss)
plt.title('Respuesta al escalon: 0.23*Mpss')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.grid(True)

# Comparacion superpuesta
plt.figure(figsize=(9, 4.5))
plt.plot(T_tf, y_tf, label='Modelo FT (Gptf)')
plt.plot(T_ss, y_ss, '--', label='Modelo EE (Mpss)')
plt.title('Comparacion de respuestas al escalon (amplitud 0.23)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.legend()
plt.grid(True)
plt.show()

# 5.2.1 y 5.2.2 con step_info
info_tf = sc.step_info(Gptf_amp)
info_ss = sc.step_info(Mpss_amp)

print('\n--- Resultados 5.2.1 y 5.2.2 ---')
print(f'Valor final FT (DC gain * 0.23): {sc.dcgain(Gptf_amp):.6f}')
print(f'Valor final EE (DC gain * 0.23): {sc.dcgain(Mpss_amp):.6f}')
print(f'Valor final esperado = 0.23*k = {amp*k:.6f}')
print(f'Tiempo de estabilizacion FT (2%): {info_tf["SettlingTime"]:.6f} s')
print(f'Tiempo de estabilizacion EE (2%): {info_ss["SettlingTime"]:.6f} s')

# Diccionarios completos por si se requieren mas metricas
print('\nStep info FT:', info_tf)
print('Step info EE:', info_ss)

## 5.3 Agregar tiempo muerto a los modelos (aproximacion de Pade)

En MATLAB se usa el comando `get` para inspeccionar la informacion de una estructura. En Python se puede usar `print(modelo)` o `vars(modelo)` para ver los campos internos.

### 5.3.1 Aproximacion de Pade

En Python no existe `InputDelay` ni `OutputDelay` directamente en `python-control`, por lo que se aproxima el tiempo muerto mediante la aproximacion de **Pade**:

$$e^{-sT_m} \approx \frac{N_{pade}(s)}{D_{pade}(s)}$$

Se usa `sc.pade(Tm, n)` que devuelve `(num_pade, den_pade)` de orden `n`. Luego se multiplica al modelo original:

$$G_{ptfm}(s) = G_0(s)\cdot G_{pade}(s), \qquad M_{pssm} = G_{ptfm}(s) \text{ convertido a EE}$$

In [ ]:
# ---- Inspeccion de modelos (equivalente a 'get' de MATLAB) ----
print('=' * 55)
print('Informacion de Gptf (Funcion de Transferencia):')
print('=' * 55)
print(Gptf)

print('\n' + '=' * 55)
print('Informacion de Mpss (Espacio de Estado):')
print('=' * 55)
print(Mpss)

# ---- Aproximacion de Pade para el tiempo muerto ----
Tm = 39.2
n_pade = 3   # orden de la aproximacion (mayor n => mejor precision)

num_pade, den_pade = sc.pade(Tm, n_pade)
Gpade = sc.tf(num_pade, den_pade)

print('\nAproximacion de Pade (orden {}):'.format(n_pade))
print(Gpade)

# ---- Modelos con tiempo muerto ----
Gptfm = Gptf * Gpade           # FT con tiempo muerto (Pade)
Mpssm = sc.ss(Mpss * Gpade)    # EE con tiempo muerto (Pade, convertido a SS)

print('\n' + '=' * 55)
print('Gptfm: FT con tiempo muerto (Pade):')
print('=' * 55)
print(Gptfm)

print('\n' + '=' * 55)
print('Mpssm: EE con tiempo muerto (Pade):')
print('=' * 55)
print(Mpssm)

### 5.3.2 Respuestas al escalon con tiempo muerto (subplot)

Se grafica en una misma figura con dos ventanas verticales la respuesta al escalon de amplitud `0.23` de cada modelo con tiempo muerto.

In [ ]:
amp = 0.23
T  = np.linspace(0, 600, 6000)

T1, y1 = sc.step_response(amp * Gptfm, T)
T2, y2 = sc.step_response(amp * Mpssm, T)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# Ventana 1 – Gptfm (FT con tiempo muerto Pade)
axes[0].plot(T1, y1, 'b', linewidth=1.5)
axes[0].set_title('Respuesta al escalon (0.23*Gptfm) — FT con tiempo muerto (Pade)')
axes[0].set_ylabel('Amplitud')
axes[0].grid(True)

# Ventana 2 – Mpssm (EE con tiempo muerto Pade)
axes[1].plot(T2, y2, 'r', linewidth=1.5)
axes[1].set_title('Respuesta al escalon (0.23*Mpssm) — EE con tiempo muerto (Pade)')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Comentario: ambas respuestas deben ser identicas (misma dinamica);
# el tiempo muerto de Tm=39.2 s se observa como retardo en el inicio de la respuesta.

### 5.3.3 Extraccion de datos al espacio de trabajo (ETPython)

Se leen los campos de las estructuras con tiempo muerto y se guardan en variables con nombres en mayuscula.

### 5.3.4 Ceros, polos, valores propios, polinomio caracteristico y ganancia DC

Se usa el modelo sin tiempo muerto para calcular las propiedades dinamicas del sistema.

In [ ]:
# =====================================================================
# 5.3.3  Extraccion de campos a variables del ETPython (mayusculas)
# =====================================================================

# --- Desde Gptfm (FT con tiempo muerto) ---
NUM_list, DEN_list = sc.tfdata(Gptfm)
NUM = np.array(NUM_list[0][0])   # coeficientes del numerador
DEN = np.array(DEN_list[0][0])   # coeficientes del denominador
TM  = Tm                          # tiempo muerto

# --- Desde Mpssm (EE con tiempo muerto) ---
A_MAT = Mpssm.A
B_MAT = Mpssm.B
C_MAT = Mpssm.C
D_MAT = Mpssm.D

print('=== Variables extraidas (5.3.3) ===')
print(f'NUM = {NUM}')
print(f'DEN = {DEN}')
print(f'TM  = {TM} s')
print(f'A =\n{A_MAT}')
print(f'B =\n{B_MAT}')
print(f'C =  {C_MAT}')
print(f'D =  {D_MAT}')

# =====================================================================
# 5.3.4  Propiedades del sistema (modelo sin tiempo muerto)
# =====================================================================
print('\n=== Propiedades del sistema (5.3.4) ===')

ceros  = sc.zeros(Gptf)
polos  = sc.poles(Gptf)
eigval = np.linalg.eigvals(Mpss.A)
p_car  = np.poly(polos)          # polinomio caracteristico (coeficientes)
dc     = float(sc.dcgain(Gptf))  # ganancia DC

print(f'Ceros del sistema:              {ceros}')
print(f'Polos del sistema:              {polos}')
print(f'Valores propios de A:           {eigval}')
print(f'Polinomio caracteristico:       {np.round(p_car, 6)}')
print(f'Ganancia DC (frecuencia cero):  {dc:.6f}')

## 5.4 Conversion entre representaciones de sistemas

En `python-control` se puede convertir entre representaciones usando las mismas funciones `sc.tf()`, `sc.ss()` y `sc.zpk()` pasando directamente un modelo como argumento.

| Comando                   | Descripcion                                        |
|---------------------------|----------------------------------------------------|
| `sc.tf(Mpss)`             | Convierte EE → Funcion de Transferencia            |
| `sc.ss(Gptf)`             | Convierte FT → Espacio de Estado                   |
| `sc.zpk(Gptf)`            | Convierte FT → Ceros-Polos-Ganancia                |
| `sc.zpk(Mpss)`            | Convierte EE → Ceros-Polos-Ganancia                |
| `sc.ss(zpk_sys)`          | Convierte ZPK → Espacio de Estado                  |
| `sc.tf(zpk_sys)`          | Convierte ZPK → Funcion de Transferencia           |
| `sc.minreal(sys)`         | Realización minima (cancela polos/ceros comunes)   |

**¿`systf` y `sysss` son iguales a `Gptf` y `Mpss`?**  
Representan el **mismo sistema** pero en forma diferente. `systf = sc.tf(Mpss)` y `Gptf` deben tener identicos numeradores y denominadores (dentro de tolerancia numerica). De forma analoga, `sysss = sc.ss(Gptf)` y `Mpss` tienen las mismas matrices A, B, C, D aunque pueden diferir en la base de realizacion interior.

**Comentario:**  
Las conversiones son equivalentes matematicamente; las diferencias numericas pequenas se deben al redondeo en punto flotante. Para confirmar equivalencia se comparan las respuestas al escalon de ambas parejas, que deben ser identicas.

In [ ]:
# =====================================================================
# 5.4  Conversion entre representaciones
# =====================================================================

# Conversiones principales solicitadas
systf  = sc.tf(Mpss)       # EE  -> FT
sysss  = sc.ss(Gptf)       # FT  -> EE
syszpk = sc.zpk(Gptf)      # FT  -> Ceros-Polos-Ganancia

print('=== systf = sc.tf(Mpss)  (EE -> FT) ===')
print(systf)

print('\n=== sysss = sc.ss(Gptf)  (FT -> EE) ===')
print(sysss)

print('\n=== syszpk = sc.zpk(Gptf)  (FT -> ZPK) ===')
print(syszpk)

# ----- Otras conversiones posibles ----
print('\n=== Otras conversiones ===')
print('ZPK -> EE :', sc.ss(syszpk))
print('ZPK -> FT :', sc.tf(syszpk))
print('EE  -> ZPK:', sc.zpk(Mpss))

# =====================================================================
# Verificacion: systf == Gptf?  sysss == Mpss?
# =====================================================================
print('\n=== ¿systf es equivalente a Gptf? ===')
num_Gptf, den_Gptf = sc.tfdata(Gptf)
num_systf, den_systf = sc.tfdata(systf)
print('num Gptf :', np.round(num_Gptf[0][0], 8))
print('num systf:', np.round(num_systf[0][0], 8))
print('den Gptf :', np.round(den_Gptf[0][0], 8))
print('den systf:', np.round(den_systf[0][0], 8))

print('\n=== Verificacion por respuesta al escalon ===')
T_ver = np.linspace(0, 500, 2000)
_, y_Gptf  = sc.step_response(Gptf,  T_ver)
_, y_systf = sc.step_response(systf, T_ver)
_, y_Mpss  = sc.step_response(Mpss,  T_ver)
_, y_sysss = sc.step_response(sysss, T_ver)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(T_ver, y_Gptf,  'b',  linewidth=2,   label='Gptf  (FT original)')
axes[0].plot(T_ver, y_systf, 'r--', linewidth=1.5, label='systf = tf(Mpss)')
axes[0].set_title('systf vs Gptf')
axes[0].set_xlabel('Tiempo (s)')
axes[0].set_ylabel('Amplitud')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(T_ver, y_Mpss,  'b',  linewidth=2,   label='Mpss  (EE original)')
axes[1].plot(T_ver, y_sysss, 'r--', linewidth=1.5, label='sysss = ss(Gptf)')
axes[1].set_title('sysss vs Mpss')
axes[1].set_xlabel('Tiempo (s)')
axes[1].set_ylabel('Amplitud')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Verificacion: conversiones producen el mismo sistema', fontsize=12)
plt.tight_layout()
plt.show()

print('\nDiferencia maxima |y_Gptf - y_systf|:', np.max(np.abs(y_Gptf - y_systf)))
print('Diferencia maxima |y_Mpss - y_sysss|:', np.max(np.abs(y_Mpss  - y_sysss)))